In [162]:
import numpy as np
import random as rd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, make_scorer, mean_absolute_error
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator

import warnings
warnings.filterwarnings('ignore')

In [163]:
# Load dataset
df = pd.read_csv("data/Train.csv")

Var = "2tdUrea"

df = df.dropna(subset=[Var], axis=0)

# splitting dataset
X_train, X_test, y_train, y_test = train_test_split(df.drop(["ID", "Yield"], axis=1), df.Yield, test_size=0.25, random_state = 1234)

# Missing values
cat_features = X_train.select_dtypes(np.object_).columns.to_list()
num_features = X_train.select_dtypes(np.number).columns.to_list()

num_imputer = SimpleImputer(strategy="mean")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[num_features] = num_imputer.fit_transform(X_train[num_features])
X_train[cat_features] = cat_imputer.fit_transform(X_train[cat_features])

X_test[num_features] = num_imputer.transform(X_test[num_features])
X_test[cat_features] = cat_imputer.transform(X_test[cat_features])

# Encoding
enc = OneHotEncoder(handle_unknown="ignore", sparse=False)
x = enc.fit_transform(X_train[cat_features])
y = enc.transform(X_test[cat_features])

X_train = pd.concat([X_train[num_features].reset_index(drop=True), pd.DataFrame(x)], axis=1)
X_test = pd.concat([X_test[num_features].reset_index(drop=True), pd.DataFrame(y)], axis=1)

# Instantiate model
model = RandomForestRegressor(random_state = 1234)

# Fit model
model.fit(X_train, y_train)

# Make predictions
preds = model.predict(X_test)

# Measure model performance
mean_squared_error(y_test, preds, squared=False)


116.8367063032672

## After Operation

In [164]:
# Sensibility analysis
X1 = X_train.copy()
X1[Var] = np.random.randint(1000,size=X_train.shape[0])

# Fit model
model.fit(X1, y_train)

# Make predictions
preds = model.predict(X_test)

# Measure model performance
mean_squared_error(y_test, preds, squared=False)

116.65320176331488

In [165]:
# Permutation method
X2 = X_train.copy()
rd.shuffle(X2[Var])

# Fit model
model.fit(X2, y_train)

# Make predictions
preds = model.predict(X_test)

# Measure model performance
mean_squared_error(y_test, preds, squared=False)

114.1651920486722

In [166]:
# Deletion method
X3 = X_train.copy()
X3.drop([Var], axis=1)

# Fit model
model.fit(X3.drop([Var], axis=1), y_train)

# Make predictions
preds = model.predict(X_test.drop([Var], axis=1))

# Measure model performance
mean_squared_error(y_test, preds, squared=False)

116.05999110828166